# Bella Italia Chatbot - Midterm Coursework

This notebook implements a food ordering assistant chatbot for Bella Italia restaurant.


In [77]:
# Standard Library Imports
import json
import os
import re
import random

# External Library Imports
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Download required NLTK data (run once, then comment out)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

print("All dependencies loaded successfully!")


All dependencies loaded successfully!


## 1. Loading Intents and Menu Data

This section loads the chatbot's intents, menu data, and restaurant information from the `intents.json` file. The data structure includes:
- **Restaurant information**: Name, hours, and address
- **Menu items**: Organized by categories (pastas, salads, drinks) with SKU codes, prices, and descriptions
- **Intents**: Patterns and responses for different user interactions

The `load_intents()` function handles file reading and error handling.


In [78]:
def load_intents(json_file_path='intents.json'):
    """
    Load intents, menu, and restaurant information from JSON file.
    
    Parameters:
    -----------
    json_file_path : str
        Path to the intents.json file (default: 'intents.json')
    
    Returns:
    --------
    dict
        Dictionary containing restaurant_info, menu, and intents
    """
    try:
        with open(json_file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
        print(f"Successfully loaded data from {json_file_path}")
        return data
    except FileNotFoundError:
        print(f"Error: File '{json_file_path}' not found.")
        return None
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON format in '{json_file_path}': {e}")
        return None

# Load the intents and menu data
data = load_intents('intents.json')

# Display what we loaded
if data:
    print(f"\nRestaurant: {data['restaurant_info']['name']}")
    print(f"Number of intents: {len(data['intents'])}")
    print(f"Menu categories: {list(data['menu'].keys())}")


Successfully loaded data from intents.json

Restaurant: Bella Italia
Number of intents: 7
Menu categories: ['pastas', 'salads', 'drinks']


## 2. Memory Structure

The chatbot needs to remember user information and their order throughout the conversation.


In [79]:
class ChatbotMemory:
    """
    Memory structure to store user information and order details.
    This allows the chatbot to remember context throughout the conversation.
    """
    
    def __init__(self):
        """Initialize empty memory structure."""
        self.user_name = None
        self.cart = []  # List of dictionaries: [{"item": "Spaghetti Carbonara", "quantity": 2, "price": 16.50}, ...]
        self.preferences = []  # List of favorite items (optional)
    
    def set_name(self, name):
        """Store the user's name."""
        self.user_name = name
    
    def get_name(self):
        """Get the user's name, or return None if not set."""
        return self.user_name
    
    def add_to_cart(self, item_name, quantity, price):
        """
        Add an item to the cart.
        
        Parameters:
        -----------
        item_name : str
            Name of the item
        quantity : int
            Number of items
        price : float
            Price per item
        """
        self.cart.append({
            "item": item_name,
            "quantity": quantity,
            "price": price
        })
    
    def get_cart(self):
        """Get the current cart."""
        return self.cart
    
    def get_cart_total(self):
        """Calculate total price of items in cart."""
        total = 0.0
        for item in self.cart:
            total += item["quantity"] * item["price"]
        return total
    
    def clear_cart(self):
        """Clear the cart (useful after checkout)."""
        self.cart = []
    
    def add_preference(self, item_name):
        """Add an item to user preferences."""
        if item_name not in self.preferences:
            self.preferences.append(item_name)
    
    def get_preferences(self):
        """Get user preferences."""
        return self.preferences
    
    def reset(self):
        """Reset all memory (useful for testing)."""
        self.user_name = None
        self.cart = []
        self.preferences = []

# Create a global memory instance
memory = ChatbotMemory()

# Test the memory structure
print("=== Testing Memory Structure ===\n")
print(f"Initial state - Name: {memory.get_name()}, Cart: {memory.get_cart()}")
memory.set_name("John")
memory.add_to_cart("Spaghetti Carbonara", 2, 16.50)
memory.add_to_cart("Caesar Salad", 1, 15.00)
print(f"\nAfter adding items:")
print(f"Cart: {memory.get_cart()}")
print(f"Total: £{memory.get_cart_total():.2f}")


=== Testing Memory Structure ===

Initial state - Name: None, Cart: []

After adding items:
Cart: [{'item': 'Spaghetti Carbonara', 'quantity': 2, 'price': 16.5}, {'item': 'Caesar Salad', 'quantity': 1, 'price': 15.0}]
Total: £48.00


## 3. Core Functions Implementation

Functions to process the loaded data and build internal structures for pattern matching and response generation.


In [80]:
def build_pattern_to_intent(data):
    """
    Build a dictionary mapping regex patterns to intent tags.
    
    Parameters:
    -----------
    data : dict
        Dictionary containing intents loaded from JSON
    
    Returns:
    --------
    dict
        Dictionary mapping patterns to intent tags
        Format: {pattern: tag, ...}
    """
    if not data or 'intents' not in data:
        return {}
    
    pattern_to_intent = {}
    
    for intent in data['intents']:
        tag = intent['tag']
        for pattern in intent['patterns']:
            pattern_to_intent[pattern] = tag
    
    return pattern_to_intent

# Test the function
if data:
    pattern_to_intent = build_pattern_to_intent(data)
    print("=== Pattern-to-Intent Dictionary ===\n")
    print(f"Total patterns: {len(pattern_to_intent)}")
    print(f"\nSample mappings:")
    for i, (pattern, tag) in enumerate(list(pattern_to_intent.items())[:5]):
        print(f"  '{pattern}' -> '{tag}'")


=== Pattern-to-Intent Dictionary ===

Total patterns: 94

Sample mappings:
  'hi' -> 'greeting'
  'hello' -> 'greeting'
  'hey' -> 'greeting'
  'good (?:morning|afternoon|evening)' -> 'greeting'
  'greeting' -> 'greeting'


In [81]:
def build_intent_to_response(data):
    """
    Build a dictionary mapping intent tags to their response lists.
    
    Parameters:
    -----------
    data : dict
        Dictionary containing intents loaded from JSON
    
    Returns:
    --------
    dict
        Dictionary mapping intent tags to response lists
        Format: {tag: [response1, response2, ...], ...}
    """
    if not data or 'intents' not in data:
        return {}
    
    intent_to_response = {}
    
    for intent in data['intents']:
        tag = intent['tag']
        intent_to_response[tag] = intent['responses']
    
    return intent_to_response

# Test the function
if data:
    intent_to_response = build_intent_to_response(data)
    print("=== Intent-to-Response Dictionary ===\n")
    print(f"Total intents: {len(intent_to_response)}")
    print(f"\nSample mappings:")
    for tag, responses in intent_to_response.items():
        print(f"  '{tag}': {len(responses)} response(s)")
        print(f"    Example: '{responses[0][:50]}...'")


=== Intent-to-Response Dictionary ===

Total intents: 7

Sample mappings:
  'greeting': 4 response(s)
    Example: 'Hello! Welcome to {restaurant_name}! How can I hel...'
  'name_collection': 4 response(s)
    Example: 'Nice to meet you, {name}! What would you like to o...'
  'menu_inquiry': 4 response(s)
    Example: 'Here's our menu, {name}:
{menu_list}

To order, pl...'
  'place_order': 4 response(s)
    Example: 'Added {quantity} {item_name} to your cart, {name}!...'
  'view_cart': 4 response(s)
    Example: '{cart_display}

Would you like to add anything els...'
  'confirm_order': 4 response(s)
    Example: 'Thank you for your order, {name}! Your order will ...'
  'restaurant_info': 4 response(s)
    Example: 'Here's our restaurant information, {name}:

{resta...'


In [82]:
def extract_menu_data(data):
    """
    Extract and format menu data from JSON for display.
    
    Parameters:
    -----------
    data : dict
        Dictionary containing menu data loaded from JSON
    
    Returns:
    --------
    dict
        Dictionary with formatted menu data
        Format: {category: [{"name": "...", "price": ..., "description": "..."}, ...], ...}
    """
    if not data or 'menu' not in data:
        return {}
    
    return data['menu']

def format_menu_for_display(menu_data):
    """
    Format menu data into a readable string for display.
    
    Parameters:
    -----------
    menu_data : dict
        Dictionary containing menu items by category
    
    Returns:
    --------
    str
        Formatted menu string ready for display
    """
    if not menu_data:
        return "Menu not available."
    
    menu_text = ""
    
    for category, items in menu_data.items():
        # Capitalize category name
        category_name = category.capitalize()
        menu_text += f"\n**{category_name}:**\n"
        
        for item in items:
            name = item['name']
            sku = item.get('sku', '')
            price = item['price']
            description = item.get('description', '')
            
            # Format: "• SKU - Item Name - £Price"
            if sku:
                menu_text += f"  • {sku} - {name} - £{price:.2f}"
            else:
                menu_text += f"  • {name} - £{price:.2f}"
            
            if description:
                menu_text += f"\n    {description}\n"
            else:
                menu_text += "\n"
    
    return menu_text

# Test the functions
if data:
    menu_data = extract_menu_data(data)
    formatted_menu = format_menu_for_display(menu_data)
    print("=== Menu Data Extraction ===\n")
    print(f"Menu categories: {list(menu_data.keys())}")
    print(f"\nFormatted menu preview:")
    print(formatted_menu[:300] + "...")


=== Menu Data Extraction ===

Menu categories: ['pastas', 'salads', 'drinks']

Formatted menu preview:

**Pastas:**
  • SC - Spaghetti Carbonara - £16.50
    Classic Roman pasta with eggs, pancetta, parmesan, and black pepper
  • PA - Penne Arrabbiata - £14.50
    Spicy tomato sauce with garlic, red chili, and fresh basil
  • FA - Fettuccine Alfredo - £17.50
    Rich creamy sauce with parmesan cheese...


## 4. Text Preprocessing

Preprocess user input to improve pattern matching reliability. This includes tokenization, punctuation removal, and lemmatization.


### Requirements

Before running this notebook, ensure you have the following installed:

**Python Standard Library:**
- `json` - For loading JSON data files
- `re` - For regular expression pattern matching
- `random` - For random response selection
- `os` - For file path operations

**External Packages:**
- `nltk` (Natural Language Toolkit) - For text preprocessing
  - Required NLTK data:
    - `punkt_tab` - Tokenizer data
    - `stopwords` - Stop words corpus
    - `wordnet` - WordNet corpus for lemmatization

**Installation:**
```bash
pip install nltk
```

The notebook will automatically download required NLTK data on first run if not already present.


In [83]:
def preprocess_input(user_input):
    """
    Preprocess user input to improve pattern matching reliability.
    Always removes stopwords (except important words) and applies lemmatization.
    
    Steps:
    1. Convert to lowercase
    2. Remove punctuation (keep apostrophes for contractions)
    3. Normalize whitespace
    4. Tokenize into words
    5. Remove stop words (keeping important words for pattern matching)
    6. Apply lemmatization to reduce words to root form
    
    Parameters:
    -----------
    user_input : str
        Raw user input string
    
    Returns:
    --------
    str
        Preprocessed input string ready for pattern matching
    """
    if not user_input:
        return ""
    
    # 1. Convert to lowercase
    processed = user_input
    
    # 2. Remove punctuation (keep apostrophes for contractions and numbers)
    processed = re.sub(r'[^\w\s\'\d]', '', processed)
    
    # 3. Normalize whitespace (remove extra spaces, tabs, newlines)
    processed = re.sub(r'\s+', ' ', processed).strip()
    
    # 4-6. Tokenize and process tokens
    try:
        # Tokenize into words
        tokens = word_tokenize(processed)
        
        # 5. Remove stop words (but keep important words for pattern matching)
        from nltk.corpus import stopwords
        stop_words = set(stopwords.words('english'))
        # Keep important words that are needed for pattern matching
        important_words = {
            'i', 'what', 'how', 'when', 'where', 'who', 'show', 'me', 'my', 'name', 'is',
            'have', 'order', 'menu', 'list', 'display', 'call', 'am', 'food', 'item',
            'option', 'dish', 'serve', 'morning', 'afternoon', 'evening', 'please',
            'this', 'see', 'want'
        }
        stop_words = stop_words - important_words
        tokens = [token for token in tokens if token not in stop_words]
        
        # 6. Apply lemmatization (always, but preserve certain words)
        lemmatizer = WordNetLemmatizer()
        # Words to preserve as-is (don't lemmatize)
        preserve_words = {'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 
                          'evening', 'morning', 'afternoon'}
        lemmatized_tokens = []
        for token in tokens:
            # Don't lemmatize preserved words
            if token.lower() in preserve_words:
                lemmatized_tokens.append(token)
                continue
            
            # Don't lemmatize if it looks like a proper noun (capitalized after lowercasing)
            # or if it's a single character
            if len(token) == 1:
                lemmatized_tokens.append(token)
                continue
            
            # Try verb first (for words like "ordering" -> "order")
            lemmatized = lemmatizer.lemmatize(token, pos='v')
            # If unchanged, try as noun
            if lemmatized == token:
                lemmatized = lemmatizer.lemmatize(token, pos='n')
            # If still unchanged, try as adjective (for words like "evening")
            if lemmatized == token:
                lemmatized = lemmatizer.lemmatize(token, pos='a')
            
            # Only use lemmatized if it's a valid word (not too short or changed drastically)
            if len(lemmatized) >= 2 and lemmatized != '':
                lemmatized_tokens.append(lemmatized)
            else:
                lemmatized_tokens.append(token)
        tokens = lemmatized_tokens
        
        # Join tokens back into string
        processed = ' '.join(tokens)
    except Exception as e:
        # If tokenization fails, return the preprocessed string without lemmatization
        print(f"Warning: Tokenization failed: {e}")
    
    return processed

# Test the preprocessing function
print("=== Testing Preprocessing Function ===\n")

test_inputs = [
    "Hello!",
    "Show me the menu, please.",
    "I'm John",
    "What do you have?",
    "I want to order 2 pizzas",
    "Ordering   multiple   items",
    "HELLO WORLD"
]

print("Original -> Preprocessed:")
for test_input in test_inputs:
    preprocessed = preprocess_input(test_input)
    print(f"  '{test_input}' -> '{preprocessed}'")


=== Testing Preprocessing Function ===

Original -> Preprocessed:
  'Hello!' -> 'Hello'
  'Show me the menu, please.' -> 'Show me menu please'
  'I'm John' -> 'I 'm John'
  'What do you have?' -> 'What have'
  'I want to order 2 pizzas' -> 'I want order 2 pizza'
  'Ordering   multiple   items' -> 'Ordering multiple item'
  'HELLO WORLD' -> 'HELLO WORLD'


## 5. Pattern Recognition (Regex)

Implement regex-based pattern matching to identify user intents and extract information like names and quantities.


In [84]:
import re

def match_pattern(user_input, pattern_to_intent_dict):
    """
    Match preprocessed user input against regex patterns and return intent tag with extracted values.
    Prioritizes more specific patterns (menu_inquiry, name_collection) over general ones (greeting).
    
    Parameters:
    -----------
    user_input : str
        Preprocessed user input string
    pattern_to_intent_dict : dict
        Dictionary mapping regex patterns to intent tags
    
    Returns:
    --------
    tuple: (intent_tag, match_info)
        intent_tag: str or None - The matched intent tag, or None if no match
        match_info: dict - Dictionary containing extracted values (name, quantity, etc.)
    """
    if not user_input or not pattern_to_intent_dict:
        return None, {}
    
    match_info = {}
    
    # Priority order: check specific intents before general ones
    # This prevents "hey show me menu" from matching "hey" first
    # Also prevents "confirm my order" from matching "my order" (view_cart) first
    priority_intents = ['menu_inquiry', 'name_collection', 'place_order', 'confirm_order', 'view_cart', 'restaurant_info']
    other_intents = ['greeting']
    
    # Group patterns by intent priority
    # Further prioritize confirm_order before view_cart to prevent "confirm my order" matching "my order"
    priority_patterns_confirm = []
    priority_patterns_other = []
    other_patterns = []
    
    for pattern, intent_tag in pattern_to_intent_dict.items():
        if intent_tag == 'confirm_order':
            priority_patterns_confirm.append((pattern, intent_tag))
        elif intent_tag in priority_intents:
            priority_patterns_other.append((pattern, intent_tag))
        else:
            other_patterns.append((pattern, intent_tag))
    
    # Try priority patterns first, with confirm_order checked before view_cart
    all_patterns = priority_patterns_confirm + priority_patterns_other + other_patterns
    
    # Try to match against each pattern
    for pattern, intent_tag in all_patterns:
        # Use case-insensitive matching (though input is already lowercase)
        # Try matching the pattern directly (patterns already contain regex constructs)
        try:
            # Handle anchored patterns (^pattern$)
            if pattern.startswith('^') and pattern.endswith('$'):
                # Full string match
                match = re.fullmatch(pattern[1:-1], user_input, re.IGNORECASE)
            elif pattern.startswith('^'):
                # Start of string match
                match = re.match(pattern[1:], user_input, re.IGNORECASE)
            elif pattern.endswith('$'):
                # End of string match
                match = re.search(pattern[:-1] + '$', user_input, re.IGNORECASE)
            else:
                # Substring match (default)
                match = re.search(pattern, user_input, re.IGNORECASE)
            
            if match:
                # Extract capture groups
                groups = match.groups()
                if groups:
                    # Store captured groups (e.g., name from "i (\\w+)")
                    match_info['captured'] = groups
                
                return intent_tag, match_info
        except re.error:
            # Skip invalid regex patterns
            continue
    
    # No pattern matched
    return None, {}

def extract_quantity(user_input):
    """
    Extract quantity (number) from user input using regex.
    
    Parameters:
    -----------
    user_input : str
        Preprocessed user input string
    
    Returns:
    --------
    int or None
        Extracted quantity, or None if no number found
    """
    # Match one or more digits
    match = re.search(r'\d+', user_input)
    if match:
        return int(match.group())
    return None

def extract_item_name(user_input, menu_data):
    """
    Extract item name from user input by matching against menu items.
    
    Parameters:
    -----------
    user_input : str
        Preprocessed user input string
    menu_data : dict
        Dictionary containing menu items by category
    
    Returns:
    --------
    str or None
        Matched item name, or None if no match found
    """
    if not menu_data:
        return None
    
    # Get all menu item names
    all_items = []
    for category, items in menu_data.items():
        for item in items:
            item_name = item['name'].lower()
            all_items.append((item_name, item['name']))  # (lowercase, original)
    
    # Try to find matching item in user input
    user_lower = user_input.lower()
    for lowercase_name, original_name in all_items:
        # Check if item name appears in user input
        # Split item name into words and check if all words appear
        item_words = lowercase_name.split()
        if all(word in user_lower for word in item_words):
            return original_name
    
    return None

# Test the pattern matching function
if data:
    pattern_to_intent = build_pattern_to_intent(data)
    
    print("=== Testing Pattern Matching ===\n")
    
    test_cases = [
        # Greeting variations
        "hello", "hi", "hey", "Hello!", "Hi there", "Hey there",
        "good morning", "good afternoon", "good evening",
        "Good Morning!", "greetings", "Hello there!",
        
        # Menu inquiry variations
        "show me menu", "Can you show me the menu?",
        "Can you please show me the menu?", "Show me what you have",
        "what do you have", "what can I order", "what's on the menu",
        "what's in the menu", "menu please", "show menu", "display menu",
        "list items", "what options do you have", "what dishes",
        "show me the food", "what do you serve", "menu",
        "I want to see the menu", "Can I see what you have?",
        
        # Name collection variations
        "I am John", "I'm John", "my name is Sarah", "I'm Sarah",
        "I am David", "my name is Michael", "call me Emma",
        "name is James", "this is Lisa", "I'm Tom", "name John",
        
        # Edge cases
        "", "   ", "hello world", "show me the menu please",
        "I'm", "menu menu menu", "HELLO", "Hello!!!",
        "Hey, can you show me the menu?"
    ]
    
    for test_input in test_cases:
        preprocessed = preprocess_input(test_input)
        intent_tag, match_info = match_pattern(preprocessed, pattern_to_intent)
        print(f"Input: '{test_input}'")
        print(f"  Preprocessed: '{preprocessed}'")
        print(f"  Matched intent: {intent_tag}")
        if match_info.get('captured'):
            print(f"  Captured: {match_info['captured']}")
        print()
    
    # Test quantity extraction
    print("=== Testing Quantity Extraction ===\n")
    quantity_tests = ["order 2 pizza", "i want 3 salads", "give me five items"]
    for test in quantity_tests:
        preprocessed = preprocess_input(test)
        quantity = extract_quantity(preprocessed)
        print(f"'{test}' -> '{preprocessed}' -> Quantity: {quantity}")


=== Testing Pattern Matching ===

Input: 'hello'
  Preprocessed: 'hello'
  Matched intent: greeting

Input: 'hi'
  Preprocessed: 'hi'
  Matched intent: greeting

Input: 'hey'
  Preprocessed: 'hey'
  Matched intent: greeting

Input: 'Hello!'
  Preprocessed: 'Hello'
  Matched intent: greeting

Input: 'Hi there'
  Preprocessed: 'Hi'
  Matched intent: greeting

Input: 'Hey there'
  Preprocessed: 'Hey'
  Matched intent: greeting

Input: 'good morning'
  Preprocessed: 'good morning'
  Matched intent: greeting

Input: 'good afternoon'
  Preprocessed: 'good afternoon'
  Matched intent: greeting

Input: 'good evening'
  Preprocessed: 'good evening'
  Matched intent: greeting

Input: 'Good Morning!'
  Preprocessed: 'Good Morning'
  Matched intent: greeting

Input: 'greetings'
  Preprocessed: 'greeting'
  Matched intent: greeting

Input: 'Hello there!'
  Preprocessed: 'Hello'
  Matched intent: greeting

Input: 'show me menu'
  Preprocessed: 'show me menu'
  Matched intent: menu_inquiry

Input: 'C

In [85]:
def format_cart_for_display(memory):
    """
    Format cart items into a readable string for display.
    
    Parameters:
    -----------
    memory : ChatbotMemory
        Memory object containing cart data
    
    Returns:
    --------
    str
        Formatted cart string ready for display
        Returns "Your cart is empty." if cart is empty
    """
    cart = memory.get_cart()
    
    if not cart:
        return "Your cart is empty."
    
    cart_text = "Your cart:\n"
    total = 0.0
    
    for item in cart:
        item_name = item['item']
        quantity = item['quantity']
        price = item['price']
        item_total = quantity * price
        total += item_total
        
        # Format: "• 2x Spaghetti Carbonara - £33.00"
        cart_text += f"  • {quantity}x {item_name} - £{item_total:.2f}\n"
    
    # Add total line
    cart_text += f"\nTotal: £{total:.2f}"
    
    return cart_text


## 6. Response Generation

Generate dynamic responses based on matched intents, with placeholder substitution for restaurant name, user name, and menu display.


In [86]:
import random

def generate_response(intent_tag, intent_to_response_dict, memory, menu_data, restaurant_info, quantity=None, item_name=None):
    """
    Generate a response for a given intent tag with placeholder substitution.
    
    Parameters:
    -----------
    intent_tag : str
        The matched intent tag (e.g., 'greeting', 'menu_inquiry', 'name_collection', 'place_order')
    intent_to_response_dict : dict
        Dictionary mapping intent tags to response lists
    memory : ChatbotMemory
        Memory object containing user name and cart
    menu_data : dict
        Menu data from JSON (used for menu_inquiry responses)
    restaurant_info : dict
        Restaurant information from JSON (contains name, hours, address)
    quantity : int, optional
        Quantity of items (for place_order intent)
    item_name : str, optional
        Name of the item (for place_order intent)
    
    Returns:
    --------
    str
        Final response string with placeholders replaced, or None if intent_tag is invalid
    """
    if not intent_tag or intent_tag not in intent_to_response_dict:
        return None
    
    # Get list of possible responses for this intent
    responses = intent_to_response_dict[intent_tag]
    
    if not responses:
        return None
    
    # Select a random response from the list
    selected_response = random.choice(responses)
    
    # Replace placeholders with actual values
    # {restaurant_name} -> restaurant name
    if '{restaurant_name}' in selected_response:
        restaurant_name = restaurant_info.get('name', 'Bella Italia')
        selected_response = selected_response.replace('{restaurant_name}', restaurant_name)
    
    # {restaurant_hours} -> restaurant hours
    if '{restaurant_hours}' in selected_response:
        restaurant_hours = restaurant_info.get('hours', 'Monday-Sunday: 11:00 AM - 10:00 PM')
        selected_response = selected_response.replace('{restaurant_hours}', restaurant_hours)
    
    # {restaurant_address} -> restaurant address
    if '{restaurant_address}' in selected_response:
        restaurant_address = restaurant_info.get('address', '123 Italian Street, London, UK')
        selected_response = selected_response.replace('{restaurant_address}', restaurant_address)
    
    # {name} -> user's name from memory, or "guest" if not set
    if '{name}' in selected_response:
        user_name = memory.get_name()
        if user_name:
            selected_response = selected_response.replace('{name}', user_name)
        else:
            # If name placeholder exists but name not set, use "guest" as fallback
            selected_response = selected_response.replace('{name}', 'guest')
    
    # {menu_list} -> formatted menu display
    if '{menu_list}' in selected_response:
        formatted_menu = format_menu_for_display(menu_data)
        selected_response = selected_response.replace('{menu_list}', formatted_menu)
    
    # {quantity} -> quantity of items (for place_order intent)
    if '{quantity}' in selected_response:
        if quantity is not None:
            selected_response = selected_response.replace('{quantity}', str(quantity))
        else:
            selected_response = selected_response.replace('{quantity}', '1')
    
    # {item_name} -> item name (for place_order intent)
    if '{item_name}' in selected_response:
        if item_name:
            selected_response = selected_response.replace('{item_name}', item_name)
        else:
            selected_response = selected_response.replace('{item_name}', 'item')
    
    # {cart_display} -> formatted cart display (for view_cart intent)
    if '{cart_display}' in selected_response:
        cart_display = format_cart_for_display(memory)
        selected_response = selected_response.replace('{cart_display}', cart_display)
    
    # {cart_total} -> total price of cart (for confirm_order intent)
    if '{cart_total}' in selected_response:
        cart_total = memory.get_cart_total()
        selected_response = selected_response.replace('{cart_total}', f'{cart_total:.2f}')
    
    return selected_response



In [87]:
# Test the response generation function
if data:
    # Build required dictionaries
    intent_to_response = build_intent_to_response(data)
    menu_data = extract_menu_data(data)
    restaurant_info = data.get('restaurant_info', {})
    
    print("=== Testing Response Generation ===\n")
    
    # Test greeting response
    print("1. Greeting Response:")
    response = generate_response('greeting', intent_to_response, memory, menu_data, restaurant_info)
    print(f"   {response}\n")
    
    # Test menu inquiry response
    print("2. Menu Inquiry Response:")
    response = generate_response('menu_inquiry', intent_to_response, memory, menu_data, restaurant_info)
    print(f"   {response[:200]}...\n")
    
    # Test name collection response (without name set)
    print("3. Name Collection Response (no name set):")
    memory_temp = ChatbotMemory()  # Fresh memory without name
    response = generate_response('name_collection', intent_to_response, memory_temp, menu_data, restaurant_info)
    print(f"   {response}\n")
    
    # Test name collection response (with name set)
    print("4. Name Collection Response (with name 'John'):")
    memory_temp.set_name("John")
    response = generate_response('name_collection', intent_to_response, memory_temp, menu_data, restaurant_info)
    print(f"   {response}\n")
    
    # Test place_order intent with quantity and item_name
    print("5. Place Order Response (with quantity and item_name):")
    response = generate_response('place_order', intent_to_response, memory, menu_data, restaurant_info, 
                                quantity=2, item_name="Spaghetti Carbonara")
    print(f"   {response}\n")
    
    # Test place_order intent with only item_name (quantity defaults to "1")
    print("6. Place Order Response (only item_name, quantity=None):")
    response = generate_response('place_order', intent_to_response, memory, menu_data, restaurant_info, 
                                quantity=None, item_name="Caesar Salad")
    print(f"   {response}\n")
    
    # Test place_order intent with only quantity (item_name defaults to "item")
    print("7. Place Order Response (only quantity, item_name=None):")
    response = generate_response('place_order', intent_to_response, memory, menu_data, restaurant_info, 
                                quantity=3, item_name=None)
    print(f"   {response}\n")
    
    # Test place_order intent with both None (both should default)
    print("8. Place Order Response (both quantity and item_name None):")
    response = generate_response('place_order', intent_to_response, memory, menu_data, restaurant_info, 
                                quantity=None, item_name=None)
    print(f"   {response}\n")
    
    # Test view_cart intent with empty cart
    print("9. View Cart Response (empty cart):")
    memory_empty = ChatbotMemory()
    response = generate_response('view_cart', intent_to_response, memory_empty, menu_data, restaurant_info)
    print(f"   {response}\n")
    
    # Test view_cart intent with items in cart
    print("10. View Cart Response (cart with items):")
    memory_with_items = ChatbotMemory()
    memory_with_items.add_to_cart("Spaghetti Carbonara", 2, 16.50)
    memory_with_items.add_to_cart("Caesar Salad", 1, 15.00)
    response = generate_response('view_cart', intent_to_response, memory_with_items, menu_data, restaurant_info)
    print(f"   {response[:150]}...\n")
    
    # Test confirm_order intent with items in cart
    print("11. Confirm Order Response (cart with items):")
    memory_confirm = ChatbotMemory()
    memory_confirm.add_to_cart("Spaghetti Carbonara", 2, 16.50)
    memory_confirm.add_to_cart("Caesar Salad", 1, 15.00)
    response = generate_response('confirm_order', intent_to_response, memory_confirm, menu_data, restaurant_info)
    print(f"   {response[:200]}...\n")
    
    # Test confirm_order intent with empty cart (should show empty cart message)
    print("12. Confirm Order Response (empty cart - will show empty message):")
    memory_empty_confirm = ChatbotMemory()
    response = generate_response('confirm_order', intent_to_response, memory_empty_confirm, menu_data, restaurant_info)
    print(f"   {response[:150]}...\n")
    
    # Test multiple calls to show randomness
    print("13. Multiple Greeting Responses (showing randomness):")
    for i in range(3):
        response = generate_response('greeting', intent_to_response, memory, menu_data, restaurant_info)
        print(f"   Response {i+1}: {response}")

=== Testing Response Generation ===

1. Greeting Response:
   Hello! Welcome to Bella Italia! How can I help you today?

Here are some things you can ask me:
  • Say 'show me menu' or 'what can I order' to see our menu
  • Say 'order [quantity] [SKU]' to add items (e.g., 'order 2 SC')
  • Say 'show cart' to view your order
  • Say 'confirm order' when you're ready to checkout
  • Say 'restaurant info' or 'what are your hours' for restaurant details
  • Tell me your name (e.g., 'I'm John') for personalized service

2. Menu Inquiry Response:
   We have:

**Pastas:**
  • SC - Spaghetti Carbonara - £16.50
    Classic Roman pasta with eggs, pancetta, parmesan, and black pepper
  • PA - Penne Arrabbiata - £14.50
    Spicy tomato sauce with garli...

3. Name Collection Response (no name set):
   Nice to meet you, guest! What would you like to order?

4. Name Collection Response (with name 'John'):
   Hello John! Welcome to Bella Italia! What would you like to order?

5. Place Order Response (

## 7. Order Management System

Helper functions for order management: SKU lookup and cart display formatting.


In [88]:
def display_welcome_message():
    """
    Display welcome message with available intents/commands.
    
    This function prints a welcome message and a list of available commands
    that users can use to interact with the chatbot. It helps users understand
    what the chatbot can do.
    
    Returns:
    --------
    None
        This function only prints to stdout and doesn't return a value.
    """
    print("Welcome to Bella Italia Chatbot!")
    print("Type 'exit' or 'quit' to end the conversation.\n")


In [89]:
def find_item_by_sku(sku, menu_data):
    """
    Find menu item by SKU code.
    
    Parameters:
    -----------
    sku : str
        SKU code (e.g., "SC", "PA", "FA")
    menu_data : dict
        Menu data from JSON (dictionary with categories as keys)
    
    Returns:
    --------
    dict or None
        Item dictionary with keys: name, sku, price, description
        Returns None if SKU not found
    """
    if not sku or not menu_data:
        return None
    
    # Convert SKU to uppercase for case-insensitive matching
    sku_upper = sku.upper()
    
    # Search through all categories
    for category, items in menu_data.items():
        for item in items:
            # Get SKU from item (handle both with and without SKU field)
            item_sku = item.get('sku', '')
            if item_sku and item_sku.upper() == sku_upper:
                # Return a copy of the item dict
                return {
                    'name': item['name'],
                    'sku': item.get('sku', ''),
                    'price': item['price'],
                    'description': item.get('description', ''),
                    'category': category
                }
    
    # SKU not found
    return None

def format_cart_for_display(memory):
    """
    Format cart items into a readable string for display.
    
    Parameters:
    -----------
    memory : ChatbotMemory
        Memory object containing cart data
    
    Returns:
    --------
    str
        Formatted cart string ready for display
        Returns "Your cart is empty." if cart is empty
    """
    cart = memory.get_cart()
    
    if not cart:
        return "Your cart is empty."
    
    cart_text = "Your cart:\n"
    total = 0.0
    
    for item in cart:
        item_name = item['item']
        quantity = item['quantity']
        price = item['price']
        item_total = quantity * price
        total += item_total
        
        # Format: "• 2x Spaghetti Carbonara - £33.00"
        cart_text += f"  • {quantity}x {item_name} - £{item_total:.2f}\n"
    
    # Add total line
    cart_text += f"\nTotal: £{total:.2f}"
    
    return cart_text

# Test the functions
if data:
    menu_data = extract_menu_data(data)
    
    print("=== Testing SKU Lookup Function ===\n")
    
    # Test valid SKUs
    test_skus = ["SC", "PA", "CS", "RW", "ES"]
    for sku in test_skus:
        item = find_item_by_sku(sku, menu_data)
        if item:
            print(f"SKU '{sku}': {item['name']} - £{item['price']:.2f}")
        else:
            print(f"SKU '{sku}': Not found")
    
    # Test invalid SKU
    print(f"\nSKU 'XX': {find_item_by_sku('XX', menu_data)}")
    
    print("\n=== Testing Cart Display Function ===\n")
    
    # Test empty cart
    memory_test = ChatbotMemory()
    print("Empty cart:")
    print(format_cart_for_display(memory_test))
    
    # Test cart with items
    print("\nCart with items:")
    memory_test.add_to_cart("Spaghetti Carbonara", 2, 16.50)
    memory_test.add_to_cart("Caesar Salad", 1, 15.00)
    memory_test.add_to_cart("Espresso", 2, 3.00)
    print(format_cart_for_display(memory_test))


=== Testing SKU Lookup Function ===

SKU 'SC': Spaghetti Carbonara - £16.50
SKU 'PA': Penne Arrabbiata - £14.50
SKU 'CS': Caesar Salad - £15.00
SKU 'RW': Italian Red Wine - £8.50
SKU 'ES': Espresso - £3.00

SKU 'XX': None

=== Testing Cart Display Function ===

Empty cart:
Your cart is empty.

Cart with items:
Your cart:
  • 2x Spaghetti Carbonara - £33.00
  • 1x Caesar Salad - £15.00
  • 2x Espresso - £6.00

Total: £54.00


In [90]:
def find_item_by_sku(sku, menu_data):
    """
    Find menu item by SKU code.
    
    Parameters:
    -----------
    sku : str
        SKU code (e.g., "SC", "PA", "FA")
    menu_data : dict
        Menu data from JSON (dictionary with categories as keys)
    
    Returns:
    --------
    dict or None
        Item dictionary with keys: name, sku, price, description
        Returns None if SKU not found
    """
    if not sku or not menu_data:
        return None
    
    # Convert SKU to uppercase for case-insensitive matching
    sku_upper = sku.upper()
    
    # Search through all categories
    for category, items in menu_data.items():
        for item in items:
            # Get SKU from item (handle both with and without SKU field)
            item_sku = item.get('sku', '')
            if item_sku and item_sku.upper() == sku_upper:
                # Return a copy of the item dict
                return {
                    'name': item['name'],
                    'sku': item.get('sku', ''),
                    'price': item['price'],
                    'description': item.get('description', ''),
                    'category': category
                }
    
    # SKU not found
    return None

def format_cart_for_display(memory):
    """
    Format cart items into a readable string for display.
    
    Parameters:
    -----------
    memory : ChatbotMemory
        Memory object containing cart data
    
    Returns:
    --------
    str
        Formatted cart string ready for display
        Returns "Your cart is empty." if cart is empty
    """
    cart = memory.get_cart()
    
    if not cart:
        return "Your cart is empty."
    
    cart_text = "Your cart:\n"
    total = 0.0
    
    for item in cart:
        item_name = item['item']
        quantity = item['quantity']
        price = item['price']
        item_total = quantity * price
        total += item_total
        
        # Format: "• 2x Spaghetti Carbonara - £33.00"
        cart_text += f"  • {quantity}x {item_name} - £{item_total:.2f}\n"
    
    # Add total line
    cart_text += f"\nTotal: £{total:.2f}"
    
    return cart_text

# Test the functions
if data:
    menu_data = extract_menu_data(data)
    
    print("=== Testing SKU Lookup Function ===\n")
    
    # Test valid SKUs
    test_skus = ["SC", "PA", "CS", "RW", "ES"]
    for sku in test_skus:
        item = find_item_by_sku(sku, menu_data)
        if item:
            print(f"SKU '{sku}': {item['name']} - £{item['price']:.2f}")
        else:
            print(f"SKU '{sku}': Not found")
    
    # Test invalid SKU
    print(f"\nSKU 'XX': {find_item_by_sku('XX', menu_data)}")
    
    print("\n=== Testing Cart Display Function ===\n")
    
    # Test empty cart
    memory_test = ChatbotMemory()
    print("Empty cart:")
    print(format_cart_for_display(memory_test))
    
    # Test cart with items
    print("\nCart with items:")
    memory_test.add_to_cart("Spaghetti Carbonara", 2, 16.50)
    memory_test.add_to_cart("Caesar Salad", 1, 15.00)
    memory_test.add_to_cart("Espresso", 2, 3.00)
    print(format_cart_for_display(memory_test))


=== Testing SKU Lookup Function ===

SKU 'SC': Spaghetti Carbonara - £16.50
SKU 'PA': Penne Arrabbiata - £14.50
SKU 'CS': Caesar Salad - £15.00
SKU 'RW': Italian Red Wine - £8.50
SKU 'ES': Espresso - £3.00

SKU 'XX': None

=== Testing Cart Display Function ===

Empty cart:
Your cart is empty.

Cart with items:
Your cart:
  • 2x Spaghetti Carbonara - £33.00
  • 1x Caesar Salad - £15.00
  • 2x Espresso - £6.00

Total: £54.00


In [91]:
def process_user_input(user_input, memory, pattern_to_intent, intent_to_response, menu_data, restaurant_info):
    """
    Process a single user input and generate appropriate response.
    This function encapsulates the core chatbot logic for processing user input.
    
    Parameters:
    -----------
    user_input : str
        Raw user input string
    memory : ChatbotMemory
        Memory object containing user name and cart
    pattern_to_intent : dict
        Dictionary mapping patterns to intent tags
    intent_to_response : dict
        Dictionary mapping intent tags to response lists
    menu_data : dict
        Menu data from JSON
    restaurant_info : dict
        Restaurant information from JSON
    
    Returns:
    --------
    tuple: (should_continue, should_exit)
        should_continue: bool - True if loop should continue, False if should skip to next iteration
        should_exit: bool - True if loop should exit (user said exit/quit)
    """
    # Check for exit conditions
    if user_input.lower() in ['exit', 'quit']:
        print("Bot: Thank you for visiting Bella Italia! Have a great day!")
        return False, True  # Don't continue, should exit
    
    # Handle empty input
    if not user_input.strip():
        print("Bot: Please enter a message or type 'exit' to quit.")
        return False, False  # Don't continue processing, but don't exit
    
    # 1. Preprocess input
    preprocessed = preprocess_input(user_input)
    
    # 2. Match pattern
    intent_tag, match_info = match_pattern(preprocessed, pattern_to_intent)
    
    # 3. Handle name extraction (if name_collection intent)
    if intent_tag == 'name_collection' and match_info.get('captured'):
        name = match_info['captured'][0]  # Extract name from capture group
        memory.set_name(name)
    
    # 3b. Handle place order intent (extract SKU and quantity, add to cart)
    quantity = None
    item_name = None
    if intent_tag == 'place_order' and match_info.get('captured'):
        captured = match_info['captured']
        # Patterns can have 1 or 2 capture groups: (quantity, SKU) or just (SKU)
        if len(captured) == 2:
            # Pattern has both quantity and SKU: "order (\\d+) ([a-zA-Z]{2})"
            quantity = int(captured[0])
            sku = captured[1]
        elif len(captured) == 1:
            # Pattern has only SKU: "order ([a-zA-Z]{2})"
            quantity = 1  # Default to 1 if not specified
            sku = captured[0]
        else:
            quantity = 1
            sku = None
        
        # Look up item by SKU
        if sku:
            item = find_item_by_sku(sku, menu_data)
            if item:
                item_name = item['name']
                # Add item to cart
                memory.add_to_cart(item_name, quantity, item['price'])
            else:
                # Invalid SKU
                print(f"Bot: I'm sorry, I couldn't find an item with SKU '{sku}'. Please check the menu and try again.")
                return False, False  # Don't continue processing, but don't exit
    
    # 3c. Handle confirm order intent (check if cart is empty)
    if intent_tag == 'confirm_order':
        # Check if cart is empty
        if not memory.get_cart():
            print("Bot: Your cart is empty. Please add some items before confirming your order.")
            return False, False  # Don't continue processing, but don't exit
    
    # 4. Generate response
    if intent_tag:
        response = generate_response(intent_tag, intent_to_response, memory, 
                                   menu_data, restaurant_info, quantity=quantity, item_name=item_name)
        if response:
            print(f"Bot: {response}")
            # Clear cart after order confirmation
            if intent_tag == 'confirm_order':
                memory.clear_cart()
        else:
            print("Bot: I'm sorry, I couldn't generate a response.")
    else:
        # Fallback for unmatched intents
        print("Bot: I'm sorry, I didn't understand that. Could you please rephrase?")
    
    return True, False  # Continue processing, don't exit


## 2. Main Loop

The main loop handles user input and terminates when the user types "exit" or "quit".


In [92]:
def main_loop():
    """
    Main conversation loop for the chatbot.
    Integrates pattern matching, response generation, and memory management.
    Takes user input and terminates when user types "exit" or "quit".
    """
    # Initialize data structures
    memory = ChatbotMemory()
    pattern_to_intent = build_pattern_to_intent(data)
    intent_to_response = build_intent_to_response(data)
    menu_data = extract_menu_data(data)
    restaurant_info = data.get('restaurant_info', {})
    
    display_welcome_message()
    
    while True:
        # Get user input
        user_input = input("You: ").strip()
        
        # Process user input
        should_continue, should_exit = process_user_input(
            user_input, memory, pattern_to_intent, intent_to_response, 
            menu_data, restaurant_info
        )
        
        if should_exit:
            break
        if not should_continue:
            continue

## Test cases
Let's simulate three different use cases
1- Customer just want information about the restaurant
2- Customer just want to check the menu
3- Customer wants to place an actual order

In [93]:
def demo_loop(test_inputs=None):
    """
    Demo version of the main loop for testing in Jupyter notebooks.
    Uses a list of test inputs instead of interactive input().
    
    Parameters:
    -----------
    test_inputs : list, optional
        List of test input strings. If None, uses default test conversation.
    """
    # Initialize data structures
    memory = ChatbotMemory()
    pattern_to_intent = build_pattern_to_intent(data)
    intent_to_response = build_intent_to_response(data)
    menu_data = extract_menu_data(data)
    restaurant_info = data.get('restaurant_info', {})
    
    # Default test conversation if none provided
    if test_inputs is None:
        test_inputs = [
            "hello",
            "I'm John",
            "show me restaurant info",
            "show me menu",
            "what can I order",
            "order 2 SC",
            "add PA",
            "I want 1 CS",
            "show cart",
            "confirm my order",
            "exit"
        ]
    
    display_welcome_message()
    
    for user_input in test_inputs:
        print(f"You: {user_input}")
        
        # Process user input
        should_continue, should_exit = process_user_input(
            user_input, memory, pattern_to_intent, intent_to_response, 
            menu_data, restaurant_info
        )
        
        if should_exit:
            break
        if not should_continue:
            continue
        
        print()  # Empty line for readability    



In [94]:
# Test Case 1: Customer just wants information about the restaurant
print("=" * 70)
print("TEST CASE 1: Customer just wants information about the restaurant")
print("=" * 70)
print()

test_case_1 = [
    "hello",
    "what are your hours?",
    "where are you located?",
    "exit"
]

demo_loop(test_case_1)

print("\n" + "=" * 70)
print()

# Test Case 2: Customer just wants to check the menu
print("TEST CASE 2: Customer just wants to check the menu")
print("=" * 70)
print()

test_case_2 = [
    "hi",
    "show me menu",
    "what can I order?",
    "exit"
]

demo_loop(test_case_2)

print("\n" + "=" * 70)
print()

# Test Case 3: Customer wants to place an actual order
print("TEST CASE 3: Customer wants to place an actual order")
print("=" * 70)
print()

test_case_3 = [
    "hello",
    "I'm Sarah",
    "show me menu",
    "order 2 SC",
    "add 1 CS",
    "show cart",
    "confirm my order",
    "exit"
]

demo_loop(test_case_3)


TEST CASE 1: Customer just wants information about the restaurant

Welcome to Bella Italia Chatbot!
Type 'exit' or 'quit' to end the conversation.

You: hello
Bot: Hello! Ready to order some delicious Italian food? What can I help you with?

Here are some things you can ask me:
  • Say 'show me menu' or 'what can I order' to see our menu
  • Say 'order [quantity] [SKU]' to add items (e.g., 'order 2 SC')
  • Say 'show cart' to view your order
  • Say 'confirm order' when you're ready to checkout
  • Say 'restaurant info' or 'what are your hours' for restaurant details
  • Tell me your name (e.g., 'I'm John') for personalized service

You: what are your hours?
Bot: Of course, guest! Here's our restaurant info:

Bella Italia
Hours: Monday-Sunday: 11:00 AM - 10:00 PM
Address: 123 Italian Street, London, UK

You: where are you located?
Bot: Sure, guest! Here are our details:

Bella Italia
Opening Hours: Monday-Sunday: 11:00 AM - 10:00 PM
Location: 123 Italian Street, London, UK

You: exit
B

## Final result
Run the cell below to test the complete chatbot flow

In [263]:
main_loop()

Welcome to Bella Italia Chatbot!
Type 'exit' or 'quit' to end the conversation.

Bot: Welcome to Bella Italia! How can I assist you today?

Here are some things you can ask me:
  • Say 'show me menu' or 'what can I order' to see our menu
  • Say 'order [quantity] [SKU]' to add items (e.g., 'order 2 SC')
  • Say 'show cart' to view your order
  • Say 'confirm order' when you're ready to checkout
  • Say 'restaurant info' or 'what are your hours' for restaurant details
  • Tell me your name (e.g., 'I'm John') for personalized service
Bot: Nice to meet you, Paulo!
Bot: Nice to meet you, Paulo! What would you like to order?
Bot: Here's what we offer, Paulo:

**Pastas:**
  • SC - Spaghetti Carbonara - £16.50
    Classic Roman pasta with eggs, pancetta, parmesan, and black pepper
  • PA - Penne Arrabbiata - £14.50
    Spicy tomato sauce with garlic, red chili, and fresh basil
  • FA - Fettuccine Alfredo - £17.50
    Rich creamy sauce with parmesan cheese and butter
  • LB - Lasagna alla Bolo